# Cài đặt thư viện


In [1]:
# Cài đặt thư viện (thêm -q để ẩn log không cần thiết)
!pip install transformers==4.37.2 accelerate==0.27.2 peft==0.8.2 datasets evaluate sentencepiece -q

import pandas as pd
import torch
import numpy as np
from datasets import Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang chạy trên thiết bị: {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 75.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.3 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba

In [2]:
import json
import pandas as pd
from datasets import Dataset

file_path = "/kaggle/input/datasets/balancelott/acos-dataset/full_data_10k.jsonl" 

processed_data = []

# Đọc file jsonl từng dòng
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
            
        item = json.loads(line.strip())
        review_text = item.get("review_text", "")
        
        if "annotations" in item and item["annotations"]:
            acos_list = []
            
            for ann in item["annotations"]:

                acos_item = {
                    "aspect_expression": ann.get("aspect_expression", "NULL"),
                    "aspect_category": ann.get("aspect_category", "NULL"),
                    "sentiment": ann.get("sentiment", "NULL")
                }
                acos_list.append(acos_item)
            
            if acos_list:
                
                input_text = f"review: {review_text}"
                

                target_text = json.dumps(acos_list, ensure_ascii=False)
                
                processed_data.append({
                    "input_text": input_text,
                    "target_text": target_text
                })

df = pd.DataFrame(processed_data)
dataset = Dataset.from_pandas(df)

# Chia tập dữ liệu (80% Train, 20% Eval) với seed=42
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Số lượng Train: {len(train_dataset)}, Số lượng Eval: {len(eval_dataset)}")
print("\n[Ví dụ Input]:", train_dataset[0]["input_text"])
print("[Ví dụ Target]:", train_dataset[0]["target_text"])

Số lượng Train: 7893, Số lượng Eval: 1974

[Ví dụ Input]: review: món gỏi cá ngon mực 1 nắng hấp dẩn
[Ví dụ Target]: [{"aspect_expression": "món gỏi cá", "aspect_category": "Food Quality", "sentiment": "positive"}, {"aspect_expression": "mực 1 nắng", "aspect_category": "Food Quality", "sentiment": "positive"}]


In [3]:
from transformers import AutoTokenizer

teacher_model_name = "VietAI/vit5-base"
tokenizer = AutoTokenizer.from_pretrained(teacher_model_name)

def preprocess_function(examples):
    # 1. Mã hóa đầu vào (câu review)
    model_inputs = tokenizer(
        examples["input_text"], 
        max_length=256, 
        truncation=True, 
        padding="max_length"
    )
    
    # 2. Mã hóa đầu ra (Chuỗi JSON ACOS)
    labels = tokenizer(
        text_target=examples["target_text"], 
        max_length=256, 
        truncation=True, 
        padding="max_length"
    )
    
    # Đổi các ID của token padding thành -100 để hàm Loss không tính toán vào những chỗ trống
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] 
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Đang mã hóa dữ liệu...")
train_dataset = train_dataset.map(preprocess_function, batched=True)
eval_dataset = eval_dataset.map(preprocess_function, batched=True)
print("Hoàn tất mã hóa!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Đang mã hóa dữ liệu...


Map:   0%|          | 0/7893 [00:00<?, ? examples/s]

Map:   0%|          | 0/1974 [00:00<?, ? examples/s]

Hoàn tất mã hóa!


In [4]:
from transformers import AutoModelForSeq2SeqLM
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang chạy trên thiết bị: {device}")

# Load mô hình ViT5 gốc
model = AutoModelForSeq2SeqLM.from_pretrained(teacher_model_name)

Đang chạy trên thiết bị: cuda


config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/904M [00:00<?, ?B/s]

In [5]:
import numpy as np
import json

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Giải mã ID thành chữ
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Hàm bóc tách JSON an toàn
    def extract_elements(json_str, task_type):
        try:
            data = json.loads(json_str)
            if not isinstance(data, list): return []
            
            elements = []
            for item in data:
                if not isinstance(item, dict): continue
                
                # In thường và xóa khoảng trắng để so sánh word-by-word chuẩn tuyệt đối
                a = str(item.get("aspect_expression", "")).strip().lower()
                c = str(item.get("aspect_category", "")).strip().lower()
                s = str(item.get("sentiment", "")).strip().lower()
                
                # Bỏ qua các trường hợp NULL
                if task_type == "aspect" and a and a != "null":
                    elements.append(a)
                elif task_type == "category" and c and c != "null":
                    elements.append(c)
                elif task_type == "sentiment" and s and s != "null":
                    elements.append(s)
                elif task_type == "combined" and a and a != "null":
                    # Gộp thành 1 khối, đúng cả 3 mới được tính
                    elements.append((a, c, s))
            return elements
        except Exception:
            return [] # Trả về rỗng nếu mô hình sinh JSON lỗi cú pháp

    # Bộ đếm cho 4 metrics
    counters = {
        "aspect": {"tp": 0, "fp": 0, "fn": 0},
        "category": {"tp": 0, "fp": 0, "fn": 0},
        "sentiment": {"tp": 0, "fp": 0, "fn": 0},
        "combined": {"tp": 0, "fp": 0, "fn": 0}
    }
    
    for pred_str, label_str in zip(decoded_preds, decoded_labels):
        for task in counters.keys():
            # Dùng set để không quan tâm thứ tự sinh ra trước/sau của mô hình
            pred_set = set(extract_elements(pred_str, task))
            gold_set = set(extract_elements(label_str, task))
            
            counters[task]["tp"] += len(pred_set & gold_set) 
            counters[task]["fp"] += len(pred_set - gold_set) 
            counters[task]["fn"] += len(gold_set - pred_set) 
            
    # Tính F1-Score
    results = {}
    for task, counts in counters.items():
        tp, fp, fn = counts["tp"], counts["fp"], counts["fn"]
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        results[f"{task}_f1"] = round(f1 * 100, 2) # Nhân 100 cho dễ nhìn (%)

    return results

In [6]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import shutil
import os

# 1. Cấu hình Trainer
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/acos_model",
    evaluation_strategy="epoch",    
    learning_rate=2e-4,             
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,             
    num_train_epochs=3,             
    predict_with_generate=True, 
    generation_max_length=128,
    fp16=True,                   
    report_to="none"                
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 2. Bắt đầu huấn luyện
print("Bắt đầu huấn luyện mô hình ACOS...")
trainer.train()

# 3. Lưu và nén mô hình ngay sau khi xong
save_dir = "/kaggle/working/acos_model_final_vit5_base"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("\nĐang nén mô hình thành file .zip...")
shutil.make_archive("/kaggle/working/acos_vit5_lbá_final", 'zip', save_dir)
print("Hoàn tất! Bạn có thể tải file 'acos_vit5_base_final.zip' ở cột bên phải.")

2026-06-05 03:34:59.840947: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780630500.061091      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780630500.123751      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780630500.640749      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780630500.640786      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780630500.640789      58 computation_placer.cc:177] computation placer alr

Bắt đầu huấn luyện mô hình ACOS...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Aspect F1,Category F1,Sentiment F1,Combined F1
1,No log,0.066168,0.000000,0.000000,0.000000,0.000000
2,0.145300,0.050059,0.000000,0.000000,0.000000,0.000000
3,0.056300,0.047530,0.000000,0.000000,0.000000,0.000000


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Đang nén mô hình thành file .zip...
Hoàn tất! Bạn có thể tải file 'acos_vit5_base_final.zip' ở cột bên phải.
